In [1]:
import vrplib
import numpy as np
import math
import tempfile
import os
from scipy.spatial.distance import cdist
import pulp
import re
import glob
import random

# **Data**

In [2]:
def read_tsp_cappart_format(file_path):
    """
    Parses the TSP/TSPTW text files from the specified directory.
    Structure:
    - n (int)
    - n*n distance matrix entries
    - n*2 time window entries
    - n x_coords (ignored)
    - n y_coords (ignored)
    """
    with open(file_path, 'r') as f:
        # split() handles all whitespace (newlines and spaces) automatically
        values = f.read().split()

    iterator = iter(values)
    
    try:
        # 1. Read Number of Nodes
        n = int(next(iterator))
        
        # 2. Read Distance Matrix (n x n)
        # The file contains a flattened list of integer distances
        c = []
        for i in range(n):
            row = []
            for j in range(n):
                val = float(next(iterator)) # Read as float first to be safe
                row.append(int(val))        # Convert to int as per your DIDP model type
            c.append(row)
            
        # The rest of the file (Time windows, coords) is ignored for pure TSP
        # but the iterator ensures we consumed exactly what we needed.
        num_locations = n
        travel_cost = c
        return num_locations, travel_cost

    except StopIteration:
        raise ValueError(f"File {file_path} ended unexpectedly.")

In [ ]:
def read_tsp_cappart_format(file_path):
    """
    Parses a TSP/TSPTW instance file.
    Format detected: 
      [N]
      [N x N Matrix]
      [N pairs of ReadyTime DueDate] (Optional)
      [Coordinates] (Ignored)
    """
    with open(file_path, 'r') as f:
        # Read all whitespace-separated tokens (handles newlines automatically)
        tokens = f.read().split()
    
    iterator = iter(tokens)
    
    try:
        # 1. Read Number of Nodes
        n = int(next(iterator))
        
        # 2. Read Distance Matrix (n x n)
        travel_cost = []
        for i in range(n):
            row = []
            for j in range(n):
                val = float(next(iterator))
                row.append(val)
            travel_cost.append(row)
            
        # 3. Read Time Windows (if available)
        # We expect n pairs of (Ready, Due)
        ready_time = []
        due_date = []
        
        # Check if we have enough tokens left for Time Windows (2 * n)
        # We convert the remaining iterator to a list to check length
        remaining_tokens = list(iterator)
        
        if len(remaining_tokens) >= 2 * n:
            tw_iterator = iter(remaining_tokens)
            for _ in range(n):
                r = float(next(tw_iterator))
                d = float(next(tw_iterator))
                ready_time.append(r)
                due_date.append(d)
        else:
            # Default if no TW found: [0, inf]
            print("  -> Warning: No Time Windows found, using defaults.")
            ready_time = [0.0] * n
            due_date = [100000.0] * n

        return n, travel_cost, ready_time, due_date

    except StopIteration:
        raise ValueError("Unexpected end of file.")
    except ValueError as e:
        raise ValueError(f"Format error: {e}")

# ==========================================
# EXECUTION LOOP
# ==========================================

folder_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\n20"

# Get all .txt files
all_files = glob.glob(os.path.join(folder_path, "*.txt"))

# Select random instances
num_instances_to_test = 1
if len(all_files) > num_instances_to_test:
    selected_files = random.sample(all_files, num_instances_to_test)
else:
    selected_files = all_files

print(f"Found {len(all_files)} files. Selected {len(selected_files)} for testing.")
print("-" * 50)

for i, file_path in enumerate(selected_files):
    instance_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(selected_files)}] Processing: {instance_name}")
    
    try:
        # --- A. Read Data ---
        # Returns: n, matrix, ready_times, due_dates
        num_locations, travel_cost, ready_time, due_date = read_tsp_cappart_format(file_path)
        
        print(f"  -> Successfully read instance.")
        print(f"  -> Nodes: {num_locations}")
        print(f"  -> Matrix Size: {len(travel_cost)}x{len(travel_cost[0])}")
        print(f"  -> Time Windows: {len(ready_time)} pairs found")
        print(f"  -> First Node TW: [{ready_time[0]}, {due_date[0]}]")
        
        # You can now pass these to your model:
        # mdl = create_tsptw_relaxed_model(num_locations, travel_cost, ready_time, due_date)
        
    except Exception as e:
        print(f"  -> ERROR processing {instance_name}: {e}")

Found 100 files. Selected 1 for testing.
--------------------------------------------------

[1/1] Processing: 26.txt
  -> Successfully read instance.
  -> Nodes: 20
  -> Matrix Size: 20x20
  -> Time Windows: 20 pairs found
  -> First Node TW: [0.0, 1000.0]


# **TSP model**

In [ ]:
def create_tsptw_relaxed_model(num_locations, travel_time, ready_time, due_date):
    """
    Creates a TSP with Time Windows (TSPTW) Linear Relaxation model.
    Uses Tight Big-M calculation based on Cordeau (Eq 7.6a).
    """
    
    # --- 0. Data Setup ---
    StartDepot = 0
    EndDepot = num_locations 
    
    # Duplicate Data for EndDepot
    aug_ready = ready_time + [ready_time[0]]
    aug_due = due_date + [due_date[0]]
    
    # Define Service Time (s_i) = 0 since it was removed from inputs
    aug_serve = [0.0] * (num_locations + 1)
    
    # Expand Travel Time Matrix
    aug_travel = [row[:] + [row[0]] for row in travel_time] 
    aug_travel.append(aug_travel[0][:]) 

    # --- 1. Sets ---
    N = range(1, num_locations) # Customers
    V = range(num_locations + 1) # All nodes
    
    # A: Arc Set 
    A = []
    for i in V:
        for j in V:
            if i == j: continue
            if i == EndDepot: continue   # Nothing leaves EndDepot
            if j == StartDepot: continue # Nothing enters StartDepot
            A.append((i,j))

    # Helper functions for flow
    def delta_plus(i): return [j for (u, j) in A if u == i]
    def delta_minus(i): return [j for (j, v) in A if v == i]

    # --- 2. Calculate Tight Big-M (M_ij) ---
    # Formula: M_ij = max(b_i + s_i + t_ij - a_j, 0) 
    big_m = {}
    for (i, j) in A:
        # b_i + s_i + t_ij - a_j
        val = aug_due[i] + aug_serve[i] + aug_travel[i][j] - aug_ready[j]
        big_m[(i,j)] = max(val, 0)
    
    # --- 3. Initialize Model ---
    mdl = pulp.LpProblem("TSPTW_Relaxed_TightM", pulp.LpMinimize)

    # --- 4. Variables ---
    x = {}
    for (i, j) in A:
        x[(i, j)] = pulp.LpVariable(f"x_{i}_{j}", 0, 1, pulp.LpContinuous)

    w = {}
    for i in V:
        w[i] = pulp.LpVariable(f"w_{i}", aug_ready[i], aug_due[i], pulp.LpContinuous)

    # --- 5. Objective ---
    mdl += pulp.lpSum(aug_travel[i][j] * x[(i, j)] for (i, j) in A), "Total_Cost"

    # --- 6. Constraints ---

    # (A) Flow Constraints
    # Depot Out
    mdl += pulp.lpSum(x[(StartDepot, j)] for j in delta_plus(StartDepot)) == 1, "Depot_Out"
    # Depot In
    mdl += pulp.lpSum(x[(i, EndDepot)] for i in delta_minus(EndDepot)) == 1, "Depot_In"
    
    # Customer Degree (1 In, 1 Out)
    for k in N:
        mdl += pulp.lpSum(x[(k, j)] for j in delta_plus(k)) == 1, f"Assign_Out_{k}"
        mdl += pulp.lpSum(x[(i, k)] for i in delta_minus(k)) == 1, f"Assign_In_{k}"

    # (B) Time Propagation with Tight Big-M 
    # w_i + s_i + t_ij - w_j <= M_ij * (1 - x_ij)
    # Only enforced if M_ij > 0
    for (i, j) in A:
        M = big_m[(i,j)]
        
        # Optimization: If M=0, the constraint is satisfied for all valid w, x.
        # So we only add it if M > 0.
        if M > 0:
            lhs = w[i] + aug_serve[i] + aug_travel[i][j] - w[j]
            rhs = M * (1 - x[(i, j)])
            mdl += lhs <= rhs, f"TimeProp_{i}_{j}"

    return mdl

# **Execution**

In [19]:
print(f"--- Building TSP Relaxation Model (n={num_locations}) ---")
mdl = create_tsptw_relaxed_model(num_locations, travel_cost, ready_time, due_date)

# 2. Solve
print("--- Solving ---")
# Using standard solver
solver = pulp.PULP_CBC_CMD(msg=False) # Or pulp.CPLEX_CMD()
mdl.solve(solver)

# 3. Output
print(f"Status: {pulp.LpStatus[mdl.status]}")
print(f"LOWER BOUND (Objective): {pulp.value(mdl.objective)}")

# Check for fractional values (Classic in LP relaxation)
print("\nVariable Values:")
for v in mdl.variables():
    if v.varValue and v.varValue > 0.01:
        print(f"{v.name} = {v.varValue}")

--- Building TSP Relaxation Model (n=20) ---
--- Solving ---
Status: Optimal
LOWER BOUND (Objective): 243.08706099999998

Variable Values:
w_1 = 178.0
w_10 = 873.0
w_11 = 1962.0
w_12 = 1570.0
w_13 = 1816.0
w_14 = 251.0
w_15 = 420.0
w_16 = 631.0
w_17 = 1147.0
w_18 = 1280.0
w_19 = 781.0
w_2 = 1393.0
w_20 = 1000.0
w_3 = 942.0
w_4 = 1670.0
w_5 = 302.0
w_6 = 1076.0
w_7 = 1187.0
w_8 = 1881.0
w_9 = 492.0
x_0_20 = 1.0
x_10_15 = 0.999499
x_11_18 = 0.998359
x_12_6 = 0.999499
x_13_1 = 0.998359
x_14_4 = 1.0
x_15_10 = 1.0
x_16_2 = 1.0
x_17_7 = 1.0
x_18_5 = 0.998359
x_19_9 = 0.999499
x_1_13 = 1.0
x_2_16 = 0.998568
x_3_8 = 1.0
x_4_14 = 0.998568
x_5_11 = 1.0
x_6_12 = 1.0
x_7_17 = 0.999054
x_8_3 = 0.999054
x_9_19 = 0.999499
